In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("dataset\jarvis_intents_2000.csv");
print( df.sample(10))

                        command           intent
1242          start my linkedin    open_linkedin
292                look this up    search_google
624           open notes please     open_notepad
1894           please terminate             exit
519         please search video   search_youtube
902    please launch calculator  open_calculator
170                 time please         get_time
294        please google search    search_google
1755  can you listen on spotify     open_spotify
756   hey jarvis notepad please     open_notepad


In [3]:
df['intent'].value_counts()

intent
get_time           200
search_google      200
search_youtube     200
open_notepad       200
open_calculator    200
open_whatsapp      200
open_linkedin      200
open_github        200
open_spotify       200
exit               200
Name: count, dtype: int64

In [4]:
df.describe()


,command,intent
count,2000,2000
unique,650,10
top,search on google,get_time
freq,14,200


In [5]:
df['intent_number'] = df['intent'].map({
    'get_time': 0,
    'search_google': 1,
    'search_youtube': 2,
    'open_notepad': 3,
    'open_calculator': 4,
    'open_whatsapp': 5,
    'open_linkedin': 6,
    'open_github': 7,
    'open_spotify': 8,
    'exit': 9
})
print( df.sample(10))

                      command          intent  intent_number
410      can you search video  search_youtube              2
676     please launch notepad    open_notepad              3
1324  please linkedin updates   open_linkedin              6
555             youtube video  search_youtube              2
1686         spotify playlist    open_spotify              8
1131   launch whatsapp for me   open_whatsapp              5
1190   start whatsapp app now   open_whatsapp              5
298             search online   search_google              1
1827      hey jarvis exit now            exit              9
1740        spotify music now    open_spotify              8


In [6]:
import spacy
from spacy.lang.en.stop_words import STOP_WORDS

stopwords = list(STOP_WORDS)
print(stopwords)


['otherwise', 'yourself', 'still', 'this', 'give', "'s", 'full', 'rather', 'of', 'various', 'down', 'from', 'latter', 'some', '’m', 'mostly', 'quite', 'on', 'hereupon', 'empty', 'beyond', 'because', 'it', 'off', 'how', 'am', 'if', 're', 'first', 'keep', "'m", 'anyone', 'part', '’s', 'often', 'whereafter', 'done', 'one', 'most', 'whereby', 'show', 'where', 'anything', 'whereas', 'used', 'except', 'thence', 'enough', 'n’t', 'not', 'another', 'cannot', 'three', 'only', 'alone', 'seems', 'could', 'above', 'same', 'been', 'along', 'thereafter', '’ll', 'serious', 'your', 'hence', 'forty', 'fifty', 'seem', 'at', '‘m', 'ours', 'nobody', 'say', 'put', 'but', 'mine', 'thereupon', 'twenty', 'thus', 'meanwhile', 'again', 'always', 'due', 'move', 'her', 'did', 'several', 'sixty', 'wherever', 'hereafter', 'with', 'four', 'n‘t', 'can', 'now', 'back', 'whom', 'made', 'seemed', 'nor', 'few', 'were', 'had', 'have', 'she', '’re', 'once', 'everything', 'here', 'besides', 'five', 'please', 'everyone', 'amo

In [7]:
df['command_without_stopwords'] = df['command'].apply(lambda x: ' '.join([word for word in x.split() if word not in stopwords]))

In [8]:
df.head(10)

,command,intent,intent_number,command_without_stopwords
0,jarvis time,get_time,0,jarvis time
1,time now please,get_time,0,time
2,what's the time,get_time,0,what's time
3,hey jarvis current time,get_time,0,hey jarvis current time
4,hey jarvis what's the time,get_time,0,hey jarvis what's time
5,what's the time,get_time,0,what's time
6,what time is it now,get_time,0,time
7,time please please,get_time,0,time
8,what's the time for me,get_time,0,what's time
9,hey jarvis what's the time,get_time,0,hey jarvis what's time


In [9]:
from spacy.lang.en.stop_words import STOP_WORDS
stopwords = list(STOP_WORDS)
df['command_without_stopwords'] = df['command'].apply(
    lambda x: ' '.join([word for word in x.split() if word.lower() not in stopwords])
)

# 4️⃣ Convert to embeddings
nlp = spacy.load('en_core_web_md')
df['command_vector'] = df['command_without_stopwords'].apply(lambda x: nlp(x).vector)

In [10]:
df.sample(10)


,command,intent,intent_number,command_without_stopwords,command_vector
384,google query,search_google,1,google query,"[-0.679885, 0.57078004, -0.32488, 0.20802, -0...."
846,launch calculator please,open_calculator,4,launch calculator,"[-0.73016, 0.31736, -0.283985, 0.37217, -0.414..."
929,please open calculator,open_calculator,4,open calculator,"[-0.63917, 0.086485006, -0.057156503, 0.27728,..."
91,check time please,get_time,0,check time,"[-0.850765, 0.196984, -0.35934502, -0.24177499..."
735,open notes please,open_notepad,3,open notes,"[-0.68894, 0.08956001, -0.1349615, 0.19561, -0..."
1598,please github repos,open_github,7,github repos,"[-0.69642496, 0.049255997, -0.44192, 0.23075, ..."
788,can you launch textpad,open_notepad,3,launch textpad,"[-0.79807496, -0.32744998, -0.14941, 0.0413800..."
1377,linkedin please now,open_linkedin,6,linkedin,"[-1.038, 1.0544, -0.16399, -0.17738, 0.25153, ..."
1152,open whatsapp,open_whatsapp,5,open whatsapp,"[-0.649765, -0.216565, 0.2390235, 0.1635975, -..."
1096,can you whatsapp please,open_whatsapp,5,whatsapp,"[-0.68697, -0.21986, 0.43781, 0.075545, -0.167..."


In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df.command_vector.values,
    df.intent_number,
    test_size=0.2,
    random_state=42
)

In [12]:
X_test.shape

(400,)

In [13]:
import numpy as np
X_train_2d = np.stack(X_train)
X_test_2d = np.stack(X_test)

In [14]:
X_train_2d

array([[-0.73016   ,  0.31736   , -0.283985  , ..., -0.03101501,
         0.2741315 , -0.145015  ],
       [-0.709615  ,  0.50777   , -0.354875  , ..., -0.23804998,
        -0.0193395 ,  0.18947649],
       [-0.76467997,  0.38349664, -0.22445333, ..., -0.20174332,
         0.22660601, -0.17488568],
       ...,
       [-0.66578   ,  0.38624   , -0.15455   , ...,  0.50593   ,
         0.59029   , -0.43495   ],
       [-0.701565  ,  0.08198425, -0.1117125 , ..., -0.15189001,
        -0.31121427,  0.40516573],
       [-0.74075496,  0.01431   ,  0.01219501, ..., -0.07066001,
        -0.2757685 ,  0.63971   ]], shape=(1600, 300), dtype=float32)

In [15]:
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report



scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_2d)
X_test_scaled = scaler.transform(X_test_2d)

base_models = [
    ('lr', LogisticRegression(max_iter=1000, solver='lbfgs', multi_class='auto')),
    ('rf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ('svm', SVC(kernel='linear', probability=True, random_state=42))
]

stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

stack_clf.fit(X_train_scaled, y_train)

y_pred = stack_clf.predict(X_test_scaled)
print("✅ Accuracy:", accuracy_score(y_test, y_pred)*100)
print("\n📊 Classification Report:\n", classification_report(y_test, y_pred))

joblib.dump(stack_clf, "jarvis_intent_model_stack.pkl")
joblib.dump(scaler, "jarvis_scaler_stack.pkl")
print("✅ Stacking model and scaler saved as 'jarvis_intent_model_stack.pkl' and 'jarvis_scaler_stack.pkl'")


✅ Accuracy: 98.25

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.84      1.00      0.91        36
           1       1.00      0.91      0.96        47
           2       1.00      1.00      1.00        46
           3       1.00      1.00      1.00        36
           4       1.00      1.00      1.00        34
           5       1.00      1.00      1.00        34
           6       1.00      1.00      1.00        33
           7       1.00      1.00      1.00        44
           8       1.00      1.00      1.00        48
           9       1.00      0.93      0.96        42

    accuracy                           0.98       400
   macro avg       0.98      0.98      0.98       400
weighted avg       0.99      0.98      0.98       400

✅ Stacking model and scaler saved as 'jarvis_intent_model_stack.pkl' and 'jarvis_scaler_stack.pkl'


In [16]:
import joblib
import spacy
from spacy.lang.en.stop_words import STOP_WORDS
import numpy as np

# 1️⃣ Load saved model and scaler
stack_clf = joblib.load("jarvis_intent_model_stack.pkl")
scaler = joblib.load("jarvis_scaler_stack.pkl")

# 2️⃣ Load spaCy model
nlp = spacy.load('en_core_web_md')

# 3️⃣ Define stopwords
stopwords = list(STOP_WORDS)

# 4️⃣ Map intent numbers back to intent names
intent_map = {
    0: 'get_time',
    1: 'search_google',
    2: 'search_youtube',
    3: 'open_notepad',
    4: 'open_calculator',
    5: 'open_whatsapp',
    6: 'open_linkedin',
    7: 'open_github',
    8: 'open_spotify',
    9: 'exit'
}

# 5️⃣ Function to preprocess and predict intent
def predict_intent(command):
    # Remove stopwords
    command_clean = ' '.join([word for word in command.split() if word.lower() not in stopwords])
    
    # Convert to embedding
    command_vector = nlp(command_clean).vector.reshape(1, -1)
    
    # Scale vector
    command_scaled = scaler.transform(command_vector)
    
    # Predict intent number
    intent_number = stack_clf.predict(command_scaled)[0]
    
    # Map number to intent name
    return intent_map[intent_number]

# 6️⃣ Test commands
test_commands = [
    "Open Spotify",
    "What time is it?",
    "Launch Notepad",
    "Search Python tutorials on Google",
    "Open my GitHub profile",
    "Exit Jarvis",
    "Tell me the current time",
    "Close the assistant"
]

# 7️⃣ Run predictions
for cmd in test_commands:
    intent_name = predict_intent(cmd)
    print(f"Command: '{cmd}' -> Predicted Intent: {intent_name}")
    print (accuracy_score)


Command: 'Open Spotify' -> Predicted Intent: open_spotify
<function accuracy_score at 0x000001EA70920A60>
Command: 'What time is it?' -> Predicted Intent: get_time
<function accuracy_score at 0x000001EA70920A60>
Command: 'Launch Notepad' -> Predicted Intent: open_notepad
<function accuracy_score at 0x000001EA70920A60>
Command: 'Search Python tutorials on Google' -> Predicted Intent: search_google
<function accuracy_score at 0x000001EA70920A60>
Command: 'Open my GitHub profile' -> Predicted Intent: open_github
<function accuracy_score at 0x000001EA70920A60>
Command: 'Exit Jarvis' -> Predicted Intent: exit
<function accuracy_score at 0x000001EA70920A60>
Command: 'Tell me the current time' -> Predicted Intent: get_time
<function accuracy_score at 0x000001EA70920A60>
Command: 'Close the assistant' -> Predicted Intent: exit
<function accuracy_score at 0x000001EA70920A60>
